**NLP LAB 2**

**Submitted by Musa Sajjad**

In [5]:
!pip install -q pypdf2 pdfplumber python-docx nltk requests pandas

In [6]:
from google.colab import files
uploaded = files.upload()  # select sample_report.pdf, doc_a.docx, doc_b.docx

Saving doc_b.docx to doc_b (1).docx
Saving doc_a.docx to doc_a (1).docx
Saving sample_report.pdf to sample_report (1).pdf


In [7]:
import re, csv, json, math
from collections import Counter
from pprint import pprint

import pdfplumber
import docx
import requests
import pandas as pd
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

PDF_PATH = "sample_report.pdf"
DOCX_A_PATH = "doc_a.docx"
DOCX_B_PATH = "doc_b.docx"

---
## Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

### (a) Extract text per page

In [8]:
with pdfplumber.open(PDF_PATH) as pdf:
    pages_text = [page.extract_text() or "" for page in pdf.pages]

print(f"Total pages extracted: {len(pages_text)}")

Total pages extracted: 7


### (b) Merge pages, clean, tokenize

In [9]:
full_text = " ".join(pages_text)

cleaned = re.sub(r'[^a-zA-Z\s]', ' ', full_text).lower()
tokens = cleaned.split()

print(f"Total token count: {len(tokens)}")

Total token count: 801


### (c) Word frequency (top 10, stopwords removed)

In [10]:
STOPWORDS_20 = {
    "the", "is", "a", "an", "and", "or", "of", "to", "in", "on",
    "for", "with", "as", "that", "this", "it", "are", "be", "by", "from"
}

freq = Counter(tok for tok in tokens if tok not in STOPWORDS_20 and len(tok) > 1)

print("Top 10 most frequent words:")
for word, count in freq.most_common(10):
    print(f"  {word}: {count}")

Top 10 most frequent words:
  data: 44
  python: 24
  science: 12
  learning: 12
  libraries: 9
  analysis: 9
  can: 9
  chapter: 7
  machine: 7
  such: 6


### (d) Keyword search across pages, sentence by sentence

In [11]:
keyword = input("Enter a keyword to search for: ").strip().lower()

matches_found = False
for page_num, text in enumerate(pages_text, start=1):
    sentences = re.split(r'(?<=[.!?])\s+', text or "")
    for sentence in sentences:
        if keyword in sentence.lower():
            print(f"[Page {page_num}] {sentence.strip()}")
            matches_found = True

if not matches_found:
    print(f'No sentences containing "{keyword}" were found.')

Enter a keyword to search for: MUSA
No sentences containing "musa" were found.


---
## Task 2: Structured Page Metadata Extraction and CSV Export

### (a) Build per-page metadata dictionaries

In [12]:
with pdfplumber.open(PDF_PATH) as pdf:
    raw_pages = [page.extract_text() or "" for page in pdf.pages]

assert len(raw_pages) >= 5, "PDF should have at least 5 pages for this task."

pages_data = []
for i, text in enumerate(raw_pages, start=1):
    word_count = len(text.split())
    char_count = len(text.replace(" ", "").replace("\n", ""))
    pages_data.append({
        "page_number": i,
        "raw_text": text,
        "word_count": word_count,
        "char_count": char_count,
    })

print("First 2 entries:")
pprint(pages_data[:2])

First 2 entries:
[{'char_count': 716,
  'page_number': 1,
  'raw_text': 'An Introduction to Data Science and Python\n'
              'Programming\n'
              'Chapter 1: What is Data Science?\n'
              'Data science is a field that combines statistics, programming, '
              'and domain knowledge to extract\n'
              'insight from data. Python has become the most popular language '
              'for data science because it is\n'
              'easy to learn and has powerful libraries. A typical data '
              'science project starts with collecting data\n'
              'from various sources such as databases, files, or web APIs. '
              'Once the data is collected, it must be\n'
              'cleaned and prepared before any analysis can begin. Data '
              'cleaning often takes more time than the\n'
              'actual analysis itself. Python provides libraries such as '
              'pandas and numpy that make data cleaning\n'
     

### (b) Highest / lowest word-count pages, average word count

In [13]:
highest_page = max(pages_data, key=lambda p: p["word_count"])
nonzero_pages = [p for p in pages_data if p["word_count"] > 0]
lowest_nonzero_page = min(nonzero_pages, key=lambda p: p["word_count"]) if nonzero_pages else None
avg_word_count = round(sum(p["word_count"] for p in pages_data) / len(pages_data), 2)

print(f"Highest word count -> Page {highest_page['page_number']} ({highest_page['word_count']} words)")
print(highest_page["raw_text"])
print()

if lowest_nonzero_page:
    print(f"Lowest non-zero word count -> Page {lowest_nonzero_page['page_number']} "
          f"({lowest_nonzero_page['word_count']} words)")
else:
    print("No pages with a non-zero word count were found.")

print(f"\nAverage word count across all pages: {avg_word_count}")

Highest word count -> Page 1 (137 words)
An Introduction to Data Science and Python
Programming
Chapter 1: What is Data Science?
Data science is a field that combines statistics, programming, and domain knowledge to extract
insight from data. Python has become the most popular language for data science because it is
easy to learn and has powerful libraries. A typical data science project starts with collecting data
from various sources such as databases, files, or web APIs. Once the data is collected, it must be
cleaned and prepared before any analysis can begin. Data cleaning often takes more time than the
actual analysis itself. Python provides libraries such as pandas and numpy that make data cleaning
much easier. Many organizations now rely on data science to make better business decisions. The
demand for data science skills continues to grow every year across many industries.

Lowest non-zero word count -> Page 3 (105 words)

Average word count across all pages: 115.43


### (c) Case-insensitive phrase search with a custom exception

In [14]:
class PhraseNotFoundError(Exception):
    pass

search_phrase = input("Enter a phrase to search for: ").strip()

try:
    phrase_lower = search_phrase.lower()
    found_any = False
    for p in pages_data:
        occurrences = p["raw_text"].lower().count(phrase_lower)
        if occurrences > 0:
            print(f"Page {p['page_number']}: {occurrences} occurrence(s)")
            found_any = True
    if not found_any:
        raise PhraseNotFoundError(f'The phrase "{search_phrase}" was not found in any page.')
except PhraseNotFoundError as e:
    print(f"Error: {e}")

Enter a phrase to search for: ALPHA
Error: The phrase "ALPHA" was not found in any page.


### (d) Write and re-read `pdf_summary.csv`

In [15]:
csv_path = "pdf_summary.csv"

with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])
    for p in pages_data:
        writer.writerow([p["page_number"], p["word_count"], p["char_count"]])

print(f"Wrote {csv_path}\n")

with open(csv_path, "r", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

Wrote pdf_summary.csv

['page_number', 'word_count', 'char_count']
['1', '137', '716']
['2', '120', '649']
['3', '105', '588']
['4', '106', '610']
['5', '107', '642']
['6', '118', '659']
['7', '115', '661']


---
## Task 3: PDF Text Preprocessing Pipeline for NLP

### (a) Clean raw text (URLs, digits, non-alpha, extra spaces)

In [16]:
with pdfplumber.open(PDF_PATH) as pdf:
    nlp_raw_text = " ".join(page.extract_text() or "" for page in pdf.pages)

char_count_before = len(nlp_raw_text)

step1 = re.sub(r'https?://\S+|www\.\S+', ' ', nlp_raw_text)      # (i) remove URLs
step2 = re.sub(r'\d+', ' ', step1)                                # (ii) remove digits
step3 = re.sub(r'[^a-zA-Z\s.]', ' ', step2)                        # (iii) keep letters, spaces, full stops
step4 = re.sub(r'\s+', ' ', step3).strip()                         # (iv) collapse spaces

cleaned_text = step4
char_count_after = len(cleaned_text)

print(f"Character count before cleaning: {char_count_before}")
print(f"Character count after cleaning: {char_count_after}")

Character count before cleaning: 5332
Character count after cleaning: 5290


### (b) Lowercase + split into sentences

In [17]:
lower_text = cleaned_text.lower()
raw_sentences = re.split(r'[.!?]', lower_text)
stripped_sentences = [s.strip() for s in raw_sentences]
sentence_count_before = len(stripped_sentences)

nonempty_sentences = [s for s in stripped_sentences if s != ""]
sentence_count_after = len(nonempty_sentences)

print(f"Sentence count before removing empty strings: {sentence_count_before}")
print(f"Sentence count after removing empty strings: {sentence_count_after}")

Sentence count before removing empty strings: 57
Sentence count after removing empty strings: 56


### (c) Tokenize, remove stopwords, filter short sentences

In [18]:
english_stopwords = set(stopwords.words('english'))

def tokenize(sentence):
    return sentence.split()

filtered_sentences = []
sentence_count_before_filter = len(nonempty_sentences)

for sentence in nonempty_sentences:
    tokens = tokenize(sentence)
    remaining = [t for t in tokens if t not in english_stopwords]
    if len(remaining) >= 5:
        filtered_sentences.append(" ".join(remaining))

sentence_count_after_filter = len(filtered_sentences)

print(f"Sentence count before stopword-length filter: {sentence_count_before_filter}")
print(f"Sentence count after stopword-length filter: {sentence_count_after_filter}")
print("\nSample sentences:")
for s in filtered_sentences[:3]:
    print(f"  - {s}")

Sentence count before stopword-length filter: 56
Sentence count after stopword-length filter: 56

Sample sentences:
  - introduction data science python programming chapter data science data science field combines statistics programming domain knowledge extract insight data
  - python become popular language data science easy learn powerful libraries
  - typical data science project starts collecting data various sources databases files web apis


### (d) Write `nlp_ready_output.txt` and re-read it

In [19]:
out_path = "nlp_ready_output.txt"

with open(out_path, "w") as f:
    for sentence in filtered_sentences:
        f.write(sentence + "\n")

with open(out_path, "r") as f:
    lines = [line.rstrip("\n") for line in f.readlines()]

longest = max(lines, key=len) if lines else ""
shortest = min(lines, key=len) if lines else ""

print(f"Total lines written: {len(lines)}")
print(f"Longest sentence ({len(longest)} chars): {longest}")
print(f"Shortest sentence ({len(shortest)} chars): {shortest}")

Total lines written: 56
Longest sentence (154 chars): introduction data science python programming chapter data science data science field combines statistics programming domain knowledge extract insight data
Shortest sentence (46 chars): data cleaning often takes time actual analysis


---
## Task 4: DOCX Paragraph Classification and Statistical Analysis

### (a) Extract paragraph text + style

In [20]:
doc_a = docx.Document(DOCX_A_PATH)

paragraphs_a = [{"text": p.text, "style": p.style.name} for p in doc_a.paragraphs]
unique_styles_a = sorted(set(p["style"] for p in paragraphs_a))

print(f"Total paragraph count: {len(paragraphs_a)}")
print(f"Unique style names: {unique_styles_a}")

Total paragraph count: 23
Unique style names: ['Heading 1', 'Heading 2', 'Normal', 'Title']


### (b) Classify into Headings / Body / Other-Empty

In [21]:
headings, body, other_empty = [], [], []

for p in paragraphs_a:
    style = p["style"]
    text = p["text"].strip()
    if style.startswith("Heading"):
        headings.append(p)
    elif style == "Normal" or "Body" in style:
        if text != "":
            body.append(p)
        else:
            other_empty.append(p)
    else:
        other_empty.append(p)

print(f"Headings: {len(headings)}")
print(f"Body: {len(body)}")
print(f"Other/Empty: {len(other_empty)}")

Headings: 7
Body: 13
Other/Empty: 3


### (c) Statistics on Body paragraphs with more than 5 words

In [22]:
qualifying_body = []
for p in body:
    word_count = len(p["text"].split())
    if word_count > 5:
        char_count = len(p["text"].replace(" ", ""))
        qualifying_body.append({**p, "word_count": word_count, "char_count": char_count})

if qualifying_body:
    highest_wc = max(qualifying_body, key=lambda p: p["word_count"])
    lowest_wc = min(qualifying_body, key=lambda p: p["word_count"])
    mean_wc = round(sum(p["word_count"] for p in qualifying_body) / len(qualifying_body), 2)

    print("Highest word count paragraph:")
    print(f"  ({highest_wc['word_count']} words) {highest_wc['text']}\n")
    print("Lowest word count paragraph (above 5 words):")
    print(f"  ({lowest_wc['word_count']} words) {lowest_wc['text']}\n")
    print(f"Mean word count across qualifying body paragraphs: {mean_wc}")
else:
    print("No body paragraphs with more than 5 words were found.")

Highest word count paragraph:
  (36 words) Data cleaning was performed to remove duplicate transactions and correct obvious data entry errors before any analysis began. Missing values in the discount column were treated as zero since no discount was applied in those cases.

Lowest word count paragraph (above 5 words):
  (18 words) Clothing sales showed strong seasonal patterns, with noticeable spikes during major holidays and clearance periods throughout the year.

Mean word count across qualifying body paragraphs: 26.08


### (d) Nested style dictionary, sorted by count

In [23]:
style_dict = {}
for p in paragraphs_a:
    style_dict.setdefault(p["style"], {"count": 0, "texts": []})
    style_dict[p["style"]]["count"] += 1
    style_dict[p["style"]]["texts"].append(p["text"])

for style in style_dict:
    if style_dict[style]["count"] > 2:
        style_dict[style]["texts"] = style_dict[style]["texts"][:2]

sorted_style_dict = dict(
    sorted(style_dict.items(), key=lambda item: item[1]["count"], reverse=True)
)

pprint(sorted_style_dict)

{'Heading 1': {'count': 4, 'texts': ['Introduction', 'Methodology']},
 'Heading 2': {'count': 3, 'texts': ['Background', 'Data Overview']},
 'Normal': {'count': 15,
            'texts': ['This report presents an analysis of retail sales data '
                      'collected over the past year. The goal of this project '
                      'is to identify trends and provide recommendations for '
                      'improving sales performance across all regions.',
                      'Retail businesses generate large volumes of '
                      'transactional data every single day of operation. '
                      'Understanding this data helps managers make informed '
                      'decisions about inventory, staffing, and marketing '
                      'strategies for the upcoming quarter.']},
 'Title': {'count': 1, 'texts': ['Project Report: Retail Sales Analysis']}}


---
## Task 5: DOCX Table Extraction and Pandas Integration

### (a) Extract all tables as lists of lists

In [24]:
tables_data = []
for table in doc_a.tables:
    table_rows = [[cell.text for cell in row.cells] for row in table.rows]
    tables_data.append(table_rows)

print(f"Total number of tables found: {len(tables_data)}")
for i, t in enumerate(tables_data, start=1):
    rows = len(t)
    cols = len(t[0]) if rows else 0
    print(f"  Table {i}: {rows} rows x {cols} columns")

Total number of tables found: 1
  Table 1: 5 rows x 3 columns


### (b) First table -> DataFrame, empty-cell counts

In [25]:
first_table = tables_data[0]
header, data_rows = first_table[0], first_table[1:]

df_table = pd.DataFrame(data_rows, columns=header)
print(df_table)
print()

empty_counts = {}
for col in df_table.columns:
    empty_counts[col] = df_table[col].apply(lambda v: v is None or str(v).strip() == "").sum()

print("Empty/None cells per column:")
pprint(empty_counts)

      Category Units Sold Revenue (USD)
0  Electronics       1240        186500
1     Clothing       3050              
2   Home Goods                    94200
3    Groceries       8600        112300

Empty/None cells per column:
{'Category': np.int64(0),
 'Revenue (USD)': np.int64(1),
 'Units Sold': np.int64(1)}


### (c) Paragraphs before the first table

In [26]:
from docx.oxml.ns import qn

pre_table_paragraphs = []
for child in doc_a.element.body:
    if child.tag == qn('w:tbl'):
        break
    if child.tag == qn('w:p'):
        text = "".join(node.text or "" for node in child.iter(qn('w:t')))
        pre_table_paragraphs.append(text)

print(f"Paragraphs before the first table: {len(pre_table_paragraphs)}")

Paragraphs before the first table: 13


### (d) Write `docx_extracted.txt`

In [27]:
extract_path = "docx_extracted.txt"

with open(extract_path, "w") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for para in pre_table_paragraphs:
        f.write(para + "\n")
    f.write("\n--- TABLE DATA ---\n")
    for row in first_table:
        f.write(" | ".join(row) + "\n")

with open(extract_path, "r") as f:
    print(f.read())

--- PRE-TABLE PARAGRAPHS ---
Project Report: Retail Sales Analysis
Introduction
This report presents an analysis of retail sales data collected over the past year. The goal of this project is to identify trends and provide recommendations for improving sales performance across all regions.
Retail businesses generate large volumes of transactional data every single day of operation. Understanding this data helps managers make informed decisions about inventory, staffing, and marketing strategies for the upcoming quarter.
Background
The retail industry has changed significantly over the last decade due to the growth of online shopping. Traditional stores now compete with online retailers on price, convenience, and customer experience.

Short note.
Methodology
Sales data was collected from five regional stores over a twelve month period. The dataset includes transaction dates, product categories, unit prices, quantities sold, and total revenue for every recorded sale.
Data cleaning was pe

---
## Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

### (a) Heading sets across both documents

In [28]:
doc_b = docx.Document(DOCX_B_PATH)

paragraphs_b = [{"text": p.text, "style": p.style.name} for p in doc_b.paragraphs]

headings_a_set = {p["text"].strip() for p in paragraphs_a if p["style"].startswith("Heading")}
headings_b_set = {p["text"].strip() for p in paragraphs_b if p["style"].startswith("Heading")}

common_headings = headings_a_set & headings_b_set
only_a_headings = headings_a_set - headings_b_set
only_b_headings = headings_b_set - headings_a_set

print(f"Common headings: {common_headings}")
print(f"Only in Document A: {only_a_headings}")
print(f"Only in Document B: {only_b_headings}")

Common headings: {'Introduction', 'Methodology', 'Findings'}
Only in Document A: {'Data Overview', 'Regional Performance', 'Background', 'Conclusion'}
Only in Document B: {'Channel Breakdown', 'Recommendations', 'Limitations', 'Appendix'}


### (b) Deduplicate Normal-style paragraphs from both docs

In [29]:
normal_a = [p["text"] for p in paragraphs_a if p["style"] == "Normal" and p["text"].strip() != ""]
normal_b = [p["text"] for p in paragraphs_b if p["style"] == "Normal" and p["text"].strip() != ""]

merged_normal = normal_a + normal_b
count_before_dedup = len(merged_normal)

seen = set()
deduped_normal = []
for text in merged_normal:
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        deduped_normal.append(text)

count_after_dedup = len(deduped_normal)

print(f"Total paragraphs before deduplication: {count_before_dedup}")
print(f"Total paragraphs after deduplication: {count_after_dedup}")

Total paragraphs before deduplication: 27
Total paragraphs after deduplication: 25


### (c) Combined word frequency (top 20, stopwords removed)

In [30]:
STOPWORDS_15 = {
    "the", "is", "a", "an", "and", "or", "of", "to", "in", "on",
    "for", "with", "as", "that", "this"
}

def tokenize_clean(text):
    cleaned = re.sub(r'[^a-zA-Z\s]', ' ', text).lower()
    return cleaned.split()

combined_counter = Counter()
for text in deduped_normal:
    tokens = [t for t in tokenize_clean(text) if t not in STOPWORDS_15 and len(t) > 1]
    combined_counter.update(tokens)

print("Top 20 most frequent words:")
for word, count in combined_counter.most_common(20):
    print(f"  {word}: {count}")

Top 20 most frequent words:
  sales: 9
  data: 9
  across: 7
  period: 6
  all: 5
  marketing: 5
  customer: 5
  during: 5
  email: 5
  analysis: 4
  retail: 4
  performance: 4
  was: 4
  total: 4
  highest: 4
  by: 4
  should: 4
  advertising: 4
  over: 3
  year: 3


### (d) Combined summary statistics

In [31]:
total_headings = len(headings_a_set) + len(headings_b_set)
total_unique_body_paragraphs = len(deduped_normal)
vocabulary_size = len(combined_counter)

paragraph_lengths = [len(text.split()) for text in deduped_normal]
avg_paragraph_length = round(sum(paragraph_lengths) / len(paragraph_lengths), 2) if paragraph_lengths else 0

longest_paragraph = max(deduped_normal, key=lambda t: len(t.split())) if deduped_normal else ""

print(f"Total headings across both documents: {total_headings}")
print(f"Total unique body paragraphs after deduplication: {total_unique_body_paragraphs}")
print(f"Total unique word types (vocabulary size): {vocabulary_size}")
print(f"Average paragraph length in words: {avg_paragraph_length}")
print(f"Longest paragraph by word count:\n  {longest_paragraph}")

Total headings across both documents: 14
Total unique body paragraphs after deduplication: 25
Total unique word types (vocabulary size): 306
Average paragraph length in words: 22.72
Longest paragraph by word count:
  Data cleaning was performed to remove duplicate transactions and correct obvious data entry errors before any analysis began. Missing values in the discount column were treated as zero since no discount was applied in those cases.


---
## Task 7: Multi-Endpoint API Data Joining and Analysis

### (a) Fetch users and posts, validate status codes

In [32]:
USERS_URL = "https://jsonplaceholder.typicode.com/users"
POSTS_URL = "https://jsonplaceholder.typicode.com/posts"

def get_json_or_raise(url):
    response = requests.get(url)
    if response.status_code != 200:
        raise ConnectionError(f"Request to {url} failed with status code {response.status_code}")
    return response.json()

users = get_json_or_raise(USERS_URL)
posts = get_json_or_raise(POSTS_URL)

print(f"Fetched {len(users)} users and {len(posts)} posts.")

Fetched 10 users and 100 posts.


### (b) Join users with their posts

In [33]:
joined = {}
for user in users:
    user_posts = [p for p in posts if p["userId"] == user["id"]]
    joined[user["name"]] = {
        "email": user["email"],
        "post_count": len(user_posts),
        "titles": [p["title"] for p in user_posts],
    }

print("First 3 users:")
for name in list(joined.keys())[:3]:
    print(f"\n{name}:")
    pprint(joined[name])

First 3 users:

Leanne Graham:
{'email': 'Sincere@april.biz',
 'post_count': 10,
 'titles': ['sunt aut facere repellat provident occaecati excepturi optio '
            'reprehenderit',
            'qui est esse',
            'ea molestias quasi exercitationem repellat qui ipsa sit aut',
            'eum et est occaecati',
            'nesciunt quas odio',
            'dolorem eum magni eos aperiam quia',
            'magnam facilis autem',
            'dolorem dolore est ipsam',
            'nesciunt iure omnis dolorem tempora et accusantium',
            'optio molestias id quia eum']}

Ervin Howell:
{'email': 'Shanna@melissa.tv',
 'post_count': 10,
 'titles': ['et ea vero quia laudantium autem',
            'in quibusdam tempore odit est dolorem',
            'dolorum ut in voluptas mollitia et saepe quo animi',
            'voluptatem eligendi optio',
            'eveniet quod temporibus',
            'sint suscipit perspiciatis velit dolorum rerum ipsa laboriosam '
            'od

### (c) User with the most posts

In [34]:
top_user_name = max(joined, key=lambda name: joined[name]["post_count"])
top_user = next(u for u in users if u["name"] == top_user_name)
top_user_info = joined[top_user_name]

print(f"Name: {top_user['name']}")
print(f"Email: {top_user_info['email']}")
print(f"Company: {top_user['company']['name']}")
print("Post titles:")
for i, title in enumerate(top_user_info["titles"], start=1):
    print(f"  {i}. {title}")

Name: Leanne Graham
Email: Sincere@april.biz
Company: Romaguera-Crona
Post titles:
  1. sunt aut facere repellat provident occaecati excepturi optio reprehenderit
  2. qui est esse
  3. ea molestias quasi exercitationem repellat qui ipsa sit aut
  4. eum et est occaecati
  5. nesciunt quas odio
  6. dolorem eum magni eos aperiam quia
  7. magnam facilis autem
  8. dolorem dolore est ipsam
  9. nesciunt iure omnis dolorem tempora et accusantium
  10. optio molestias id quia eum


### (d) Filter by company.bs 'synergies' or city length > 8

In [35]:
matching_users = []
for user in users:
    bs_match = "synergies" in user["company"]["bs"].lower()
    city_match = len(user["address"]["city"]) > 8
    if bs_match or city_match:
        matching_users.append(user)

for user in matching_users:
    name = user["name"]
    city = user["address"]["city"]
    post_count = joined[name]["post_count"]
    print(f"{name} | {city} | posts: {post_count}")

Leanne Graham | Gwenborough | posts: 10
Ervin Howell | Wisokyburgh | posts: 10
Clementine Bauch | McKenziehaven | posts: 10
Patricia Lebsack | South Elvis | posts: 10
Chelsey Dietrich | Roscoeview | posts: 10
Mrs. Dennis Schulist | South Christy | posts: 10
Kurtis Weissnat | Howemouth | posts: 10
Nicholas Runolfsdottir V | Aliyaview | posts: 10
Glenna Reichert | Bartholomebury | posts: 10
Clementina DuBuque | Lebsackbury | posts: 10


---
## Task 8: To-Do Completion Rate Analysis Across Users

### (a) Fetch and validate todos

In [36]:
TODOS_URL = "https://jsonplaceholder.typicode.com/todos"

response = requests.get(TODOS_URL)
todos = None

if response.status_code != 200:
    print(f"Error: request failed with status code {response.status_code}")
else:
    parsed = response.json()
    if not isinstance(parsed, list):
        print("Error: expected a list of todo records.")
    elif len(parsed) == 0:
        print("Error: no todo records were returned.")
    else:
        todos = parsed
        print(f"Total todo records retrieved: {len(todos)}")

Total todo records retrieved: 200


### (b) Group todos by user, verify totals

In [37]:
todo_summary = {}
for todo in todos:
    uid = todo["userId"]
    todo_summary.setdefault(uid, {"total": 0, "completed": 0, "incomplete": 0})
    todo_summary[uid]["total"] += 1
    if todo["completed"]:
        todo_summary[uid]["completed"] += 1
    else:
        todo_summary[uid]["incomplete"] += 1

for uid in sorted(todo_summary):
    stats = todo_summary[uid]
    check = (stats["completed"] + stats["incomplete"]) == stats["total"]
    print(f"User {uid}: {stats}  ->  consistent: {check}")

User 1: {'total': 20, 'completed': 11, 'incomplete': 9}  ->  consistent: True
User 2: {'total': 20, 'completed': 8, 'incomplete': 12}  ->  consistent: True
User 3: {'total': 20, 'completed': 7, 'incomplete': 13}  ->  consistent: True
User 4: {'total': 20, 'completed': 6, 'incomplete': 14}  ->  consistent: True
User 5: {'total': 20, 'completed': 12, 'incomplete': 8}  ->  consistent: True
User 6: {'total': 20, 'completed': 6, 'incomplete': 14}  ->  consistent: True
User 7: {'total': 20, 'completed': 9, 'incomplete': 11}  ->  consistent: True
User 8: {'total': 20, 'completed': 11, 'incomplete': 9}  ->  consistent: True
User 9: {'total': 20, 'completed': 8, 'incomplete': 12}  ->  consistent: True
User 10: {'total': 20, 'completed': 12, 'incomplete': 8}  ->  consistent: True


### (c) Completion rates: highest, lowest, average

In [38]:
completion_rates = {
    uid: round(stats["completed"] / stats["total"] * 100, 1)
    for uid, stats in todo_summary.items()
}

highest_uid = max(completion_rates, key=completion_rates.get)
lowest_uid = min(completion_rates, key=completion_rates.get)
overall_average = round(sum(completion_rates.values()) / len(completion_rates), 1)

print(f"Highest completion rate: User {highest_uid} -> {completion_rates[highest_uid]}%")
print(f"Lowest completion rate: User {lowest_uid} -> {completion_rates[lowest_uid]}%")
print(f"Overall average completion rate: {overall_average}%")

Highest completion rate: User 5 -> 60.0%
Lowest completion rate: User 4 -> 30.0%
Overall average completion rate: 45.0%


### (d) Formatted leaderboard table

In [39]:
users_lookup_resp = requests.get(USERS_URL)
users_lookup_resp.raise_for_status()
user_name_lookup = {u["id"]: u["name"] for u in users_lookup_resp.json()}

ranking = sorted(completion_rates.items(), key=lambda item: item[1], reverse=True)

header = f"{'Rank'.ljust(6)}{'Name'.ljust(22)}{'Total Tasks'.ljust(14)}{'Completed'.ljust(12)}{'Completion Rate %'}"
print(header)
print("-" * len(header))

for rank, (uid, rate) in enumerate(ranking, start=1):
    name = user_name_lookup.get(uid, f"User {uid}")
    stats = todo_summary[uid]
    print(
        f"{str(rank).ljust(6)}{name.ljust(22)}{str(stats['total']).ljust(14)}"
        f"{str(stats['completed']).ljust(12)}{rate}"
    )

Rank  Name                  Total Tasks   Completed   Completion Rate %
-----------------------------------------------------------------------
1     Chelsey Dietrich      20            12          60.0
2     Clementina DuBuque    20            12          60.0
3     Leanne Graham         20            11          55.0
4     Nicholas Runolfsdottir V20            11          55.0
5     Kurtis Weissnat       20            9           45.0
6     Ervin Howell          20            8           40.0
7     Glenna Reichert       20            8           40.0
8     Clementine Bauch      20            7           35.0
9     Patricia Lebsack      20            6           30.0
10    Mrs. Dennis Schulist  20            6           30.0


---
## Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

### (a) Flatten user records

In [40]:
def flatten_user(user: dict) -> dict:
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"],
    }

flattened_users = [flatten_user(u) for u in users]
pprint(flattened_users)

[{'city': 'Gwenborough',
  'company_catchphrase': 'Multi-layered client-server neural-net',
  'company_name': 'Romaguera-Crona',
  'email': 'Sincere@april.biz',
  'id': 1,
  'lat': '-37.3159',
  'lng': '81.1496',
  'name': 'Leanne Graham',
  'username': 'Bret',
  'zipcode': '92998-3874'},
 {'city': 'Wisokyburgh',
  'company_catchphrase': 'Proactive didactic contingency',
  'company_name': 'Deckow-Crist',
  'email': 'Shanna@melissa.tv',
  'id': 2,
  'lat': '-43.9509',
  'lng': '-34.4618',
  'name': 'Ervin Howell',
  'username': 'Antonette',
  'zipcode': '90566-7771'},
 {'city': 'McKenziehaven',
  'company_catchphrase': 'Face to face bifurcated interface',
  'company_name': 'Romaguera-Jacobson',
  'email': 'Nathan@yesenia.net',
  'id': 3,
  'lat': '-68.6102',
  'lng': '-47.0653',
  'name': 'Clementine Bauch',
  'username': 'Samantha',
  'zipcode': '59590-4157'},
 {'city': 'South Elvis',
  'company_catchphrase': 'Multi-tiered zero tolerance productivity',
  'company_name': 'Robel-Corkery'

### (b) DataFrame, distance from origin

In [41]:
df_users = pd.DataFrame(flattened_users)

df_users["lat"] = df_users["lat"].astype(float)
df_users["lng"] = df_users["lng"].astype(float)

df_users["distance_from_origin"] = df_users.apply(
    lambda row: math.sqrt(row["lat"] ** 2 + row["lng"] ** 2), axis=1
)

pd.set_option("display.max_columns", None)
print(df_users)

farthest_user = df_users.loc[df_users["distance_from_origin"].idxmax()]
print(f"\nUser with greatest distance from origin: {farthest_user['name']} "
      f"({farthest_user['distance_from_origin']:.4f})")

   id                      name          username                      email  \
0   1             Leanne Graham              Bret          Sincere@april.biz   
1   2              Ervin Howell         Antonette          Shanna@melissa.tv   
2   3          Clementine Bauch          Samantha         Nathan@yesenia.net   
3   4          Patricia Lebsack          Karianne  Julianne.OConner@kory.org   
4   5          Chelsey Dietrich            Kamren   Lucio_Hettinger@annie.ca   
5   6      Mrs. Dennis Schulist  Leopoldo_Corkery    Karley_Dach@jasper.info   
6   7           Kurtis Weissnat      Elwyn.Skiles     Telly.Hoeger@billy.biz   
7   8  Nicholas Runolfsdottir V     Maxime_Nienow       Sherwood@rosamond.me   
8   9           Glenna Reichert          Delphine    Chaim_McDermott@dana.io   
9  10        Clementina DuBuque    Moriah.Stanton     Rey.Padberg@karina.biz   

             city     zipcode      lat       lng        company_name  \
0     Gwenborough  92998-3874 -37.3159   81.149

### (c) Hemisphere classification

In [42]:
northern = df_users[df_users["lat"] > 0]
southern = df_users[df_users["lat"] < 0]

print("Northern Hemisphere users:")
print(list(northern["name"]))

print("\nSouthern Hemisphere users:")
print(list(southern["name"]))

north_avg_distance = northern["distance_from_origin"].mean() if not northern.empty else 0
south_avg_distance = southern["distance_from_origin"].mean() if not southern.empty else 0

print(f"\nNorthern average distance: {north_avg_distance:.4f}")
print(f"Southern average distance: {south_avg_distance:.4f}")

if north_avg_distance > south_avg_distance:
    print("Northern Hemisphere has the higher average distance from origin.")
else:
    print("Southern Hemisphere has the higher average distance from origin.")

Northern Hemisphere users:
['Patricia Lebsack', 'Kurtis Weissnat', 'Glenna Reichert']

Southern Hemisphere users:
['Leanne Graham', 'Ervin Howell', 'Clementine Bauch', 'Chelsey Dietrich', 'Mrs. Dennis Schulist', 'Nicholas Runolfsdottir V', 'Clementina DuBuque']

Northern average distance: 123.5833
Southern average distance: 84.3162
Northern Hemisphere has the higher average distance from origin.


### (d) Comment counts per post

In [43]:
COMMENTS_URL = "https://jsonplaceholder.typicode.com/comments"

posts_resp = requests.get(POSTS_URL)
comments_resp = requests.get(COMMENTS_URL)
posts_resp.raise_for_status()
comments_resp.raise_for_status()

all_posts = posts_resp.json()
all_comments = comments_resp.json()

comment_counts = Counter(c["postId"] for c in all_comments)

post_comment_list = [
    (p["id"], p["title"][:40], comment_counts.get(p["id"], 0))
    for p in all_posts
]
post_comment_list.sort(key=lambda t: t[2], reverse=True)

print("Top 5 most-commented posts:")
for post_id, title, count in post_comment_list[:5]:
    print(f"  Post {post_id}: {title!r} -> {count} comments")

print("\nBottom 5 least-commented posts:")
for post_id, title, count in post_comment_list[-5:]:
    print(f"  Post {post_id}: {title!r} -> {count} comments")

Top 5 most-commented posts:
  Post 1: 'sunt aut facere repellat provident occae' -> 5 comments
  Post 2: 'qui est esse' -> 5 comments
  Post 3: 'ea molestias quasi exercitationem repell' -> 5 comments
  Post 4: 'eum et est occaecati' -> 5 comments
  Post 5: 'nesciunt quas odio' -> 5 comments

Bottom 5 least-commented posts:
  Post 96: 'quaerat velit veniam amet cupiditate aut' -> 5 comments
  Post 97: 'quas fugiat ut perspiciatis vero provide' -> 5 comments
  Post 98: 'laboriosam dolor voluptates' -> 5 comments
  Post 99: 'temporibus sit alias delectus eligendi p' -> 5 comments
  Post 100: 'at nam consequatur ea labore ea harum' -> 5 comments
